In [1]:
import numpy as np
import pandas as pd
import nltk 
import string

In [2]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Abhi0072/HateXplain")

Repo card metadata block was not found. Setting CardData to empty.


In [3]:
# Access the 'train' split
train_dataset = ds['train']

# Convert to pandas DataFrame
df = train_dataset.to_pandas()

df.sample(20)



,text,label
11634,<user> you should not care if your wifes son g...,offensive
18470,starbucks everybody remember that in muslim co...,offensive
13526,<user> i heard asians got some fire though,offensive
8166,<number> spartans killed over <number> sandnig...,normal
4011,yeah the audio is killer 😂 he actually tells h...,offensive
1466,<user> <user> <user> prime raped dich,normal
18872,one of my favorite pastimes is mansplaining to...,offensive
6660,<user> <user> it hard for some to wrap their m...,normal
7636,so i had the radio on and a long commercial ca...,offensive
19077,this whole trump hates women thing is the bigg...,normal


# Basic Preprocessing

In [4]:
import nltk
import string
import re

In [5]:
from nltk.corpus import stopwords
stop = set(stopwords.words('english'))


In [6]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [7]:
def transform(text):
    text = re.sub(r'<user>', '', text)
    text = text.lower()
    tokens = nltk.word_tokenize(text)

    tokens = [word for word in tokens if word.isalnum()]
    tokens = [word for word in tokens if word not in stop]
    tokens = [ps.stem(word) for word in tokens]

    return " ".join(tokens)

In [8]:
text = "I loved the lectures on yt they were really great"
transform(text)

'love lectur yt realli great'

In [9]:
df['trans_text'] = df['text'].apply(transform)
df.head()

,text,label,trans_text
0,i dont think im getting my baby them white 9 h...,normal,dont think im get babi white 9 two white j nik...
1,we cannot continue calling ourselves feminists...,normal,continu call feminist right womxn arent addres...
2,nawt yall niggers ignoring me,normal,nawt yall nigger ignor
3,<user> i am bit confused coz chinese ppl can n...,hatespeech,bit confus coz chines ppl access twitter thn c...
4,this bitch in whataburger eating a burger with...,hatespeech,bitch whataburg eat burger top bun hold palm h...


In [10]:
from sklearn.preprocessing import LabelEncoder
ln = LabelEncoder()
df['label'] = ln.fit_transform(df['label'])

# Text Preprocessing

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

In [12]:
df.shape

(20148, 3)

In [13]:
# cv = CountVectorizer()
tfi = TfidfVectorizer(max_features=2000)

In [15]:
Xtfi = tfi.fit_transform(df['trans_text']).toarray()
Xtfi.shape

(20148, 2000)

In [16]:
y = df['label']

In [18]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(Xtfi,y,random_state=20,test_size=0.4)
X_train.shape

(12088, 2000)

In [19]:
from sklearn.metrics import accuracy_score,precision_score,confusion_matrix

In [20]:
from sklearn.linear_model import LogisticRegression
lgr = LogisticRegression()
lgr.fit(X_train,y_train)
y_pred = lgr.predict(X_test)
print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))


acc: 0.5906947890818859
precision (macro): 0.5849946636663784
confusion_matrix:
 [[1495  576  384]
 [ 262 2430  540]
 [ 398 1139  836]]


In [21]:
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import MultinomialNB
gnb = GaussianNB()
bnb = BernoulliNB()
mnb = MultinomialNB()
from sklearn.metrics import classification_report

In [114]:
gnb.fit(X_train,y_train)
y_pred=gnb.predict(X_test)
print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

acc: 0.43970223325062036
precision (macro): 0.4369010508736681
confusion_matrix:
 [[573 288 384]
 [358 755 475]
 [372 381 444]]
              precision    recall  f1-score   support

           0       0.44      0.46      0.45      1245
           1       0.53      0.48      0.50      1588
           2       0.34      0.37      0.36      1197

    accuracy                           0.44      4030
   macro avg       0.44      0.44      0.44      4030
weighted avg       0.45      0.44      0.44      4030



In [22]:
mnb.fit(X_train,y_train)
y_pred=mnb.predict(X_test)
print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

acc: 0.5647642679900744
precision (macro): 0.565632172360692
confusion_matrix:
 [[1305  862  288]
 [ 242 2664  326]
 [ 410 1380  583]]
              precision    recall  f1-score   support

           0       0.67      0.53      0.59      2455
           1       0.54      0.82      0.65      3232
           2       0.49      0.25      0.33      2373

    accuracy                           0.56      8060
   macro avg       0.57      0.53      0.52      8060
weighted avg       0.56      0.56      0.54      8060



In [23]:
bnb.fit(X_train,y_train)
y_pred=bnb.predict(X_test)
print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

acc: 0.5890818858560795
precision (macro): 0.5789664722427176
confusion_matrix:
 [[1507  505  443]
 [ 323 2298  611]
 [ 480  950  943]]
              precision    recall  f1-score   support

           0       0.65      0.61      0.63      2455
           1       0.61      0.71      0.66      3232
           2       0.47      0.40      0.43      2373

    accuracy                           0.59      8060
   macro avg       0.58      0.57      0.57      8060
weighted avg       0.58      0.59      0.58      8060



In [24]:
from sklearn.svm import LinearSVC

clf = LinearSVC()
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)
print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

acc: 0.5698511166253102
precision (macro): 0.558570459818131
confusion_matrix:
 [[1499  550  406]
 [ 329 2269  634]
 [ 479 1069  825]]
              precision    recall  f1-score   support

           0       0.65      0.61      0.63      2455
           1       0.58      0.70      0.64      3232
           2       0.44      0.35      0.39      2373

    accuracy                           0.57      8060
   macro avg       0.56      0.55      0.55      8060
weighted avg       0.56      0.57      0.56      8060



In [25]:
from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)
y_pred=rfc.predict(X_test)
print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

acc: 0.6104218362282878
precision (macro): 0.609678507410255
confusion_matrix:
 [[1539  619  297]
 [ 237 2582  413]
 [ 389 1185  799]]
              precision    recall  f1-score   support

           0       0.71      0.63      0.67      2455
           1       0.59      0.80      0.68      3232
           2       0.53      0.34      0.41      2373

    accuracy                           0.61      8060
   macro avg       0.61      0.59      0.59      8060
weighted avg       0.61      0.61      0.60      8060



In [26]:
from sklearn.ensemble import AdaBoostClassifier
adb = AdaBoostClassifier()
adb.fit(X_train, y_train)
y_pred=adb.predict(X_test)
print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

acc: 0.5426799007444169
precision (macro): 0.6424310960641791
confusion_matrix:
 [[1032 1351   72]
 [ 124 3017   91]
 [ 170 1878  325]]
              precision    recall  f1-score   support

           0       0.78      0.42      0.55      2455
           1       0.48      0.93      0.64      3232
           2       0.67      0.14      0.23      2373

    accuracy                           0.54      8060
   macro avg       0.64      0.50      0.47      8060
weighted avg       0.63      0.54      0.49      8060



In [27]:
from xgboost import XGBClassifier
xgb_clf = XGBClassifier(
    objective='multi:softmax',   
    num_class=3,                 
    eval_metric='mlogloss',      
    use_label_encoder=False,     
    n_estimators=150,            
    max_depth=6,                 
    learning_rate=0.1,
    random_state=123
)
xgb_clf.fit(X_train, y_train)
y_pred = xgb_clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

c:\Users\KIIT0001\miniconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [12:12:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.6048387096774194
precision (macro): 0.6134942876341031

Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.58      0.65      2455
           1       0.57      0.82      0.67      3232
           2       0.53      0.34      0.41      2373

    accuracy                           0.60      8060
   macro avg       0.61      0.58      0.58      8060
weighted avg       0.61      0.60      0.59      8060


Confusion Matrix:
 [[1432  719  304]
 [ 193 2641  398]
 [ 327 1244  802]]


# Applying Word2vec

In [28]:
import os 
import gensim

In [29]:
df['trans_text']

0        dont think im get babi white 9 two white j nik...
1        continu call feminist right womxn arent addres...
2                                   nawt yall nigger ignor
3        bit confus coz chines ppl access twitter thn c...
4        bitch whataburg eat burger top bun hold palm h...
                               ...                        
20143    ur still twitter tell carlton said alcohol dru...
20144    first got said hate trump consid troll peopl y...
20145    macht der moslem wenn der zion gegen seinen pr...
20146    aw look world demograph asian fuck everywher a...
20147    jewish globalist elit import million muslim mu...
Name: trans_text, Length: 20148, dtype: object

In [30]:
def getcorpus(text):
    corpus = []
    for word in text.split():
        corpus.append(word)
    
    return corpus


corpus = df['trans_text'].apply(getcorpus)
newcorpus = []

for sentence in corpus:
    filtered_sentence = [word for word in sentence if len(word) > 1]
    newcorpus.append(filtered_sentence)
    
newcorpus = [s for s in newcorpus if len(s) > 0]
print(newcorpus)

[['dont', 'think', 'im', 'get', 'babi', 'white', 'two', 'white', 'nike', 'even', 'touch'], ['continu', 'call', 'feminist', 'right', 'womxn', 'arent', 'address', 'ye', 'sexual', 'offenc', 'public', 'list', 'tran', 'lesbian', 'bisexu', 'queer', 'womxn', 'abl', 'enter', 'inform', 'report', 'sheet', 'gender', 'forum'], ['nawt', 'yall', 'nigger', 'ignor'], ['bit', 'confus', 'coz', 'chines', 'ppl', 'access', 'twitter', 'thn', 'ching', 'chong', 'use', 'think', 'pakistani'], ['bitch', 'whataburg', 'eat', 'burger', 'top', 'bun', 'hold', 'palm', 'hate', 'white', 'bitch'], ['laura', 'loomer', 'rape', 'scream', 'disgust', 'kike', 'languag', 'said', 'must', 'extermin', 'goyim', 'laura', 'loomer', 'loomerg'], ['end', 'nigger', 'traine', 'doctor', 'speak', 'properli', 'lack', 'basic', 'knowledg', 'biolog', 'truli', 'scari', 'public', 'knew'], ['nog', 'jew', 'dyke', 'enrich'], ['guilti', 'proven', 'innoc', 'unless', 'jew', 'nigger', 'lover'], ['tire', 'support', 'abort', 'moral', 'standpoint', 'wire',

Training Word2vec

In [31]:
from gensim.models import Word2Vec

In [32]:
model = Word2Vec(sentences=newcorpus,vector_size=300,window=5,min_count=2,workers=4,sg=1)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [33]:
model.wv.most_similar('black')

[('hispan', 0.9158720970153809),
 ('latino', 0.8859251141548157),
 ('color', 0.884352445602417),
 ('brown', 0.8840596675872803),
 ('okay', 0.8830147981643677),
 ('poc', 0.879932701587677),
 ('rais', 0.876706600189209),
 ('rich', 0.8744259476661682),
 ('privileg', 0.873762845993042),
 ('higher', 0.8736110925674438)]

In [110]:
model.corpus_count

20147

Making the df for word2vec

In [44]:
def avgword2vec(doc):
    vectors = [model.wv[word] for word in doc if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)


In [41]:
from tqdm import tqdm

In [47]:
X = []
for i in tqdm(range(len(newcorpus))):
    X.append(avgword2vec(newcorpus[i]))

100%|██████████| 20147/20147 [00:01<00:00, 15700.41it/s]


In [49]:
len(X)

20147

In [111]:
X_new = []
y_new = []

for doc, label in zip(newcorpus, y):
    vec = avgword2vec(doc)
    if not np.all(vec == 0):   # or len(doc) > 0
        X_new.append(vec)
        y_new.append(label)

X_new = np.array(X_new)
y_new = np.array(y_new)

In [113]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_new)


In [69]:
X_new.shape

(20144, 200)

In [70]:
y_new.shape

(20144,)

In [71]:
from sklearn.model_selection import train_test_split

In [114]:
X_train,X_test,y_train,y_test= train_test_split(X_scaled,y_new,test_size=0.2,random_state=20)
X_train

array([[-0.5661449 , -0.39534426,  0.32831675, ..., -0.579573  ,
        -0.31860372, -0.8875425 ],
       [ 4.1741204 , -2.762674  , -0.19098447, ...,  1.5907291 ,
         0.05022929,  0.9440617 ],
       [-0.43219414,  1.0565358 ,  0.09394506, ..., -1.6332339 ,
        -0.6388716 ,  0.25873017],
       ...,
       [ 0.15814346,  0.31176636, -0.31372812, ..., -0.64887375,
        -1.6441637 , -0.08618639],
       [-1.3023161 ,  0.3283553 ,  0.14056022, ..., -0.5995761 ,
        -0.40877607,  0.385848  ],
       [ 2.638554  ,  0.8701934 ,  1.2134669 , ...,  2.0248792 ,
         0.26572925,  2.475033  ]], dtype=float32)

In [90]:
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier

In [84]:
from sklearn.metrics import accuracy_score,precision_score,classification_report,confusion_matrix

In [115]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_new)

clf = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    n_jobs=-1
)

clf.fit(X_scaled, y_new)

print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))

c:\Users\KIIT0001\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


acc: 0.469843633655994
precision (macro): 0.449825348019736
confusion_matrix:
 [[ 502  584  100]
 [ 238 1246  167]
 [ 205  842  145]]


In [116]:
X_train,X_test,y_train,y_test= train_test_split(X_new,y_new,test_size=0.2,random_state=20)
gnb.fit(X_train,y_train)
y_pred=gnb.predict(X_test)
print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

acc: 0.4003474807644577
precision (macro): 0.4041288798237764
confusion_matrix:
 [[889 165 132]
 [872 555 224]
 [692 331 169]]
              precision    recall  f1-score   support

           0       0.36      0.75      0.49      1186
           1       0.53      0.34      0.41      1651
           2       0.32      0.14      0.20      1192

    accuracy                           0.40      4029
   macro avg       0.40      0.41      0.37      4029
weighted avg       0.42      0.40      0.37      4029



In [117]:
X_train,X_test,y_train,y_test= train_test_split(X_new,y_new,test_size=0.2,random_state=20)
bnb.fit(X_train,y_train)
y_pred=bnb.predict(X_test)
print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

acc: 0.4380739637627203
precision (macro): 0.4156244847653356
confusion_matrix:
 [[761 297 128]
 [603 834 214]
 [522 500 170]]
              precision    recall  f1-score   support

           0       0.40      0.64      0.50      1186
           1       0.51      0.51      0.51      1651
           2       0.33      0.14      0.20      1192

    accuracy                           0.44      4029
   macro avg       0.42      0.43      0.40      4029
weighted avg       0.43      0.44      0.41      4029



In [118]:
from sklearn.ensemble import RandomForestClassifier
X_train,X_test,y_train,y_test= train_test_split(X_new,y_new,test_size=0.2,random_state=20)
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)
y_pred=rfc.predict(X_test)
print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

acc: 0.467361628195582
precision (macro): 0.4494429467626049
confusion_matrix:
 [[ 533  521  132]
 [ 293 1140  218]
 [ 244  738  210]]
              precision    recall  f1-score   support

           0       0.50      0.45      0.47      1186
           1       0.48      0.69      0.56      1651
           2       0.38      0.18      0.24      1192

    accuracy                           0.47      4029
   macro avg       0.45      0.44      0.43      4029
weighted avg       0.45      0.47      0.44      4029



In [120]:
X_train

array([[ 0.01817735,  0.09749767, -0.05809145, ..., -0.00113984,
         0.11602888, -0.11426906],
       [ 0.17161158, -0.02052573, -0.07141415, ...,  0.06526658,
         0.12844652, -0.04867405],
       [ 0.02251311,  0.16988125, -0.06410427, ..., -0.03337952,
         0.10524631, -0.07321774],
       ...,
       [ 0.04162132,  0.13275072, -0.07456315, ..., -0.00326029,
         0.07140077, -0.08557019],
       [-0.00565124,  0.13357776, -0.06290835, ..., -0.00175189,
         0.11299302, -0.06866529],
       [ 0.12190795,  0.16059114, -0.03538287, ...,  0.07855061,
         0.13570184,  0.00615442]], dtype=float32)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
X_train,X_test,y_train,y_test= train_test_split(X_new,y_new,test_size=0.2,random_state=20)
adb = AdaBoostClassifier()
adb.fit(X_train, y_train)
y_pred=adb.predict(X_test)
print("acc:",accuracy_score(y_test,y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("confusion_matrix:\n",confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

acc: 0.46785802928766446
precision (macro): 0.44930167582878977
confusion_matrix:
 [[ 528  610   48]
 [ 295 1269   87]
 [ 261  843   88]]
              precision    recall  f1-score   support

           0       0.49      0.45      0.47      1186
           1       0.47      0.77      0.58      1651
           2       0.39      0.07      0.12      1192

    accuracy                           0.47      4029
   macro avg       0.45      0.43      0.39      4029
weighted avg       0.45      0.47      0.41      4029



In [121]:
from xgboost import XGBClassifier
xgb_clf = XGBClassifier(
    objective='multi:softmax',   
    num_class=3,                 
    eval_metric='mlogloss',      
    use_label_encoder=False,     
    n_estimators=150,            
    max_depth=6,                 
    learning_rate=0.1,
    random_state=123
)
xgb_clf.fit(X_train, y_train)
y_pred = xgb_clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

c:\Users\KIIT0001\miniconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [18:22:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.462397617274758
precision (macro): 0.4438301580933244

Classification Report:
               precision    recall  f1-score   support

           0       0.49      0.46      0.47      1186
           1       0.48      0.64      0.55      1651
           2       0.36      0.22      0.27      1192

    accuracy                           0.46      4029
   macro avg       0.44      0.44      0.43      4029
weighted avg       0.45      0.46      0.45      4029


Confusion Matrix:
 [[ 541  468  177]
 [ 300 1064  287]
 [ 258  676  258]]


In [1]:
text = " really fun day library Met some cool people"

df_test = pd.DataFrame({'text': [text]})
# Transform with TF-IDF
X_test = tfi.transform(df_test['text'])

# Predict with your trained model
y_pred = lgr.predict(X_test)

print(y_pred)

NameError: name 'pd' is not defined